# Train models and create submissions

Run STEP 01–03 first, using the commands in `../README.md`.
Select a Python kernel with the project installed and the packages in `requirements-model.txt`.

Set `ACTION` in the cell below:

- `"train"`: train once (or reuse a completed matching run), then create the ranking submission.
- `"ranking"`: use saved models to create only the ranking submission.
- `"final"`: use saved models to create `merry-llama_final.parquet`; no training.

A fresh training run can take several hours. For the completed run, choose `"final"`.
All feature and model logic lives in the package; this notebook only launches the scripts.
Files are generated locally. Uploading to the competition bucket is a separate action.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ACTION = "train"  # "train", "ranking", or "final"
RUN_ID = "v55_reproduction"  # Keep this value to use the existing saved models.

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Open this notebook from the project notebooks folder.")
if ACTION not in {"train", "ranking", "final"}:
    raise ValueError('ACTION must be "train", "ranking", or "final".')

script = "train.py" if ACTION == "train" else "predict.py"
command = [sys.executable, "-u", str(ROOT / "scripts" / script),
           "--root", str(ROOT), "--run-id", RUN_ID]
if ACTION != "train":
    command += ["--dataset", ACTION]

LOG = ROOT / "logs" / ("step04_run.log" if ACTION == "train" else f"predict_{ACTION}.log")
LOG.parent.mkdir(parents=True, exist_ok=True)
print("Python:", sys.executable)
print("Project:", ROOT)
print("Action:", ACTION)
with LOG.open("a", encoding="utf-8") as log:
    process = subprocess.Popen(
        command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace",
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        lock = ROOT / "models" / RUN_ID / "RUNNING.lock"
        if lock.exists() and lock.read_text().strip() == str(process.pid):
            lock.unlink()
        raise
print(f"\nExit code: {code} | Log: {LOG}")
if code != 0:
    raise RuntimeError(f"{ACTION} failed. Read the log: {LOG}")
